# D5 · Roads and networks: what the flood cut

> **Time:** about 3 hours  
> **You need:** Parts A, B and C, and Lessons D1 to D4, especially functions (A5), `try`/`except` (A6), `assert` (A12), `to_crs` and lengths in UTM 45N (B4), `groupby` (A9), intersections and buffers (B2, B8, B9), map elements (B11), `features.shapes` (C7) and the flood mask of D3.  
> **By the end you can:**
> 1. load **OpenStreetMap** roads as a **graph** of nodes and edges, turn it into GeoDataFrames, and measure road length by type, junctions, dead ends and **connected components**,
> 2. find the **shortest route** between two places with osmnx and networkx, after checking that each place snapped to a nearby node,
> 3. overlay the roads with D3's flood polygons, remove the roads that cross them, and measure the **disruption**: detours, places with no route, and **accessibility** from Melamchi, as a stated scenario with its sensitivity.

D3 mapped where the ground along the Melamchi river changed between October 2020 and November 2021. For the people of the valley, one of the first questions after a flood is a different one: **can we still get through?** Roads in Himalayan valleys often follow the river, because the valley floor is the easiest ground to build on. That is also the ground a flood or a debris flow sweeps.

This lesson answers the question with a **network**: the roads of the valley from **OpenStreetMap**, stored as a graph that a computer can search for routes. You will put D3's change area on top of it, take out the roads that run through it, and see which places can still reach which.

Two honest warnings come first, and the lesson returns to them:

- The roads are **today's** OpenStreetMap, downloaded in 2026. The change area comes from images of **2020 and 2021**. Roads may have been rebuilt, moved or removed since the flood, on the ground or on the map.
- Removing roads is a **scenario**: *if* every road through the change area were closed, what would follow? The images cannot tell you which roads really were closed, or for how long.

Everything loads from `../data`; nothing here needs the internet. This lesson replaces the older notebook *5(a) Accessing OSM data in Python*.

*Map data © OpenStreetMap contributors, available under the Open Database Licence (ODbL).* Wherever a map shows roads, rivers or places from OpenStreetMap, it carries this credit. Maps that use D3's change area also carry the Sentinel-2 credit.

**Counting lines:** lines that only set up or show a plot (`fig, ax = plt.subplots()`, `plt.show()`) do not count toward the Q1/Q2/Q3 line counts.

**If results look strange** (a name that is not defined, numbers that differ from the text), use **Kernel → Restart Kernel and Run All Cells**. This lesson removes roads from copies of the network; an old copy from an earlier attempt can linger in memory.

**How to practise:** after every step there are three questions. **Q1** needs one line of code, **Q2** two lines, **Q3** about three. Type your answer in the empty cell under each question, run it, then open **Show solution** to check. Try first, even if you are not sure: struggling for a few minutes is how the skill grows.

<!-- pygile-nav -->
**PyGILE Course** · lesson 38 of 39 · [Course home](../README.md)  
← Previous: [D4 · Land-cover classification: forest, fields, sediment and water](D04_Land_cover_classification.ipynb) · Next: [D6 · Image time series and data cubes: five years of the Melamchi valley](D06_Image_time_series_and_data_cubes.ipynb) →

## 1. OpenStreetMap: a map made by volunteers

**OpenStreetMap** (OSM) is a free map of the world, drawn by volunteers. Anyone can add a road, a river or a village, often by tracing satellite images or from a GPS track.

Every feature carries **tags**: pairs of **key=value**. A road might have `highway=secondary` and `name=Melamchi Road`; a village `place=village` and `name=मेलम्ची`. The key `highway` holds the road **type**, from big to small: `trunk`, `primary`, `secondary`, `tertiary`, `unclassified` (minor public roads), `residential`, `service` and `track` (rough roads, often unpaved).

Because volunteers draw it, **completeness varies**. Towns and main roads are usually well mapped. Rural tracks may be missing, unfinished or out of date. Treat OSM like any data source: useful, checkable, never perfect.

`../data/osm_melamchi.gpkg` holds three layers for the Part D box: `roads`, `rivers` and `places`. Each GeoPackage column is one tag (B3). Names are mostly in **Nepali script** (`name`); `name:en` holds an English name where a volunteer added one.

**Credit:** OSM data are open under the **ODbL** licence, which asks you to credit "© OpenStreetMap contributors".

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import networkx as nx
import osmnx as ox

DATA = Path("..") / "data"
OUT = Path("..") / "outputs"
OUT.mkdir(exist_ok=True)
OSM_CREDIT = "Map data © OpenStreetMap contributors, ODbL."

places = gpd.read_file(DATA / "osm_melamchi.gpkg", layer="places")
print(places.shape, places.crs)
print(places["place"].value_counts())
places.head(3)

In [ ]:
named = places[places["name:en"].notna()].set_index("name:en")    # places with an English name (A9)
print(len(named), "of", len(places), "places have an English name")

town = named.loc["Melamchi"]
print(town["name"], "|", town["place"], "|", round(town.geometry.x, 4), "°E", round(town.geometry.y, 4), "°N")

In [ ]:
rivers = gpd.read_file(DATA / "osm_melamchi.gpkg", layer="rivers")
print(rivers["waterway"].value_counts())

named_utm = named.to_crs(32645)                   # UTM 45N, metres (B4)
box = named_utm.total_bounds                      # west, south, east, north of the places
rivers_utm = gpd.clip(rivers.to_crs(32645), [box[0] - 1000, box[1] - 1000, box[2] + 1000, box[3] + 1000])   # B7

fig, ax = plt.subplots(figsize=(7, 9))
rivers_utm.plot(ax=ax, color="steelblue", linewidth=0.7)
rivers_utm[rivers_utm["waterway"] == "river"].plot(ax=ax, color="steelblue", linewidth=2)
named_utm.plot(ax=ax, color="black", markersize=12)
for name, point in named_utm.geometry.items():
    ax.annotate(name, (point.x, point.y), xytext=(3, 3), textcoords="offset points", fontsize=7)
ax.set_title("Rivers and named places of the Melamchi valley (OpenStreetMap)")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.ticklabel_format(style="plain", useOffset=False)
fig.text(0.01, 0.02, OSM_CREDIT, fontsize=8)
plt.show()

The `places` layer has **60** points: 43 hamlets, 8 villages, 8 suburbs and **one town**. Only 22 of them have an English name; the others have only a Nepali name or no name at all. That is typical volunteered data: what exists depends on who mapped it.

The town is **Melamchi** (`मेलम्ची`), at about **85.5761° E, 27.8293° N**. This lesson measures distances from this point. The maps of D1 and D3 mark Melamchi at the same position.

In the map (`fig` is the whole figure, `ax` the one panel of axes inside it; `ax=ax` tells each `.plot` to draw in that panel, B11), Melamchi sits where two rivers meet: one from the north-west, one from the north-east. Thick lines are tagged `waterway=river`, thin ones `stream`. OSM's river lines did not stop at the box: `features_from_bbox` returns whole features that **touch** the box, so they ran far beyond it. `gpd.clip` (B7) cut them to the area of the places plus 1 km.

**Try this at home when you are online.** This is how the data were downloaded (`../data/get_extra_data.py` holds the full version; you do not need it for this lesson):

```python
import osmnx as ox
bbox = (85.48, 27.80, 85.62, 27.98)                       # west, south, east, north in degrees
roads = ox.graph_from_bbox(bbox, custom_filter='["highway"~"secondary|tertiary|unclassified|track"]')
ox.save_graphml(roads, "my_roads.graphml")
places = ox.features_from_bbox(bbox, tags={"place": ["town", "village", "hamlet"]})
```

*Map data © OpenStreetMap contributors, available under the Open Database Licence (ODbL).*

### Practice

**Q1 · one line:** How many places of each type have an **English** name? Use `value_counts` on the `place` column of `named`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
named["place"].value_counts()
```

12 hamlets, 8 suburbs, 1 village and 1 town. All 8 suburbs have an English name, but only 12 of the 43 hamlets: volunteers named the bigger places first.
</details>

**Q2 · two lines:** Which rivers have a name in OSM? Print the `value_counts` of the `name` column of the clipped `rivers_utm`, and the number of river and stream lines with **no** name.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(rivers_utm["name"].value_counts())
print(rivers_utm["name"].isna().sum(), "lines without a name")
```

Four names: "Indrawati river" (4 lines), "Melamchi River", "Melamchi Khola" and "Fadung Khola" (*khola* means river or stream in Nepali). One river carries two names, "Melamchi River" and "Melamchi Khola", on different stretches. 124 lines, nearly all the small streams, have no name. To select a river by name you would first have to clean the names, as with Chitwan and Chitawan in B6.
</details>

**Q3 · three lines:** Earlier versions of the course marked Melamchi at Wikipedia's coordinates, 27°50′N 85°34′E, rounded to **27.83° N, 85.56° E**. How far is that point from the OSM town point? Make a GeoSeries of the rounded point, convert both to UTM 45N and print the distance in km, rounded to 2 decimals.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
rounded = gpd.GeoSeries(gpd.points_from_xy([85.56], [27.83]), crs="EPSG:4326").to_crs(32645)
osm_town = named_utm.loc["Melamchi"].geometry
print(round(rounded.iloc[0].distance(osm_town) / 1000, 2))
```

About 1.59 km. Coordinates rounded to whole minutes of arc (1′ of latitude is about 1.85 km) can be off by a kilometre or more. Wikipedia gives the centre of the whole **municipality**, a large area; OSM marks the **town** itself. Two honest sources, two different questions.
</details>

## 2. Roads as a graph: nodes and edges

A road map as a picture is lines. A road map for **routing** is a **graph**, a word from mathematics for a set of points and the links between them:

- **Nodes** are the points: junctions, dead ends and the ends of roads.
- **Edges** are the links: a stretch of road from one node to the next, with attributes such as `length` (metres) and `highway`.

The Python library **networkx** (imported as `nx`) works with graphs of any kind. **osmnx** (imported as `ox`) builds networkx graphs from OSM and adds geography: every node has `x` (longitude) and `y` (latitude), and every edge a `geometry`.

`ox.load_graphml(path)` reads a graph saved as **GraphML**, a text format for graphs. The graph you get is a **MultiDiGraph**:

- **Di** (directed): each edge goes **from** node `u` **to** node `v`. A two-way road is stored **twice**, once in each direction, so a router can respect one-way streets.
- **Multi**: two nodes can be linked by more than one edge (two different roads between the same two junctions). The third number of an edge, `key`, tells them apart: 0, 1, …

`ox.graph_to_gdfs(G)` turns the graph into two GeoDataFrames, `nodes` and `edges`, so all of Part B works on them.

**Predict:** the graph has 854 nodes. Will it have more edges than nodes, or fewer?

In [ ]:
G = ox.load_graphml(DATA / "osm_melamchi_roads.graphml")
print(type(G).__name__, "|", len(G.nodes), "nodes |", len(G.edges), "edges")
print(G.graph)

node_id = list(G.nodes)[0]
print("\none node:", node_id, G.nodes[node_id])

u, v, key = list(G.edges(keys=True))[0]
edge_data = G.edges[u, v, key]
print("one edge:", u, "->", v, "key", key)
print({name: value for name, value in edge_data.items() if name != "geometry"})     # A12's dict comprehension
print("the same road backwards exists too:", G.has_edge(v, u))

In [ ]:
nodes, edges = ox.graph_to_gdfs(G)
print("nodes:", nodes.shape, "| edges:", edges.shape, "|", edges.crs)
print("edge index:", edges.index.names)
print(edges["oneway"].value_counts())
edges[["highway", "name", "length", "oneway"]].head(4)

The graph has **854 nodes** and **1,942 edges**, more edges than nodes. `G.graph` holds facts about the whole graph: its CRS (EPSG:4326, degrees) and the date osmnx made it, in **2026**. Remember that date: this is the road map as volunteers had drawn it five years after the flood.

A node is a small dictionary: `y` and `x` (latitude and longitude) and `street_count`, how many streets meet there. An edge is a bigger one: the OSM way number (`osmid`), the road type, a reference number (`ref`), `oneway`, `reversed` and `length` in metres. Its `geometry` (left out of the print) is the road's shape as a LineString (B2); osmnx stores a geometry only for edges that are not straight.

Every one of the 1,942 edges has `oneway` `False`: every road here is two-way, so each one is stored twice. `G.has_edge(v, u)` confirms it for the first edge: the road from `v` back to `u` exists as well. The next step shows what that does to a sum of lengths.

The `edges` GeoDataFrame has a three-level index, `u`, `v` and `key`: the start node, the end node and the number of the edge between them.

### Practice

**Q1 · one line:** How many edges start at node `node_id`? `G.out_degree(node_id)` counts them.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
G.out_degree(node_id)
```

3 edges leave this node, and 3 arrive (`G.in_degree`), because every road is two-way. It is a junction where three roads meet, as its `street_count` of 3 says. (This first node happens to be the junction nearest to Timbu, a village in the north of the valley.)
</details>

**Q2 · two lines:** Which edge is the **longest**? Use `.idxmax()` (A11) on the `length` column to get its index, then print its road type and its length in km, rounded to 2 decimals.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
longest = edges["length"].idxmax()
print(edges.loc[longest, "highway"], round(edges.loc[longest, "length"] / 1000, 2))
```

An unclassified road of about 10.84 km. osmnx **simplifies** the graph: it removes every node where nothing happens (no junction, no dead end) and joins the pieces, so one edge can be a long, winding road between two junctions. `idxmax` returns the index label, here a tuple (u, v, key).
</details>

**Q3 · three lines:** Some edges hold a **list** of road types, where osmnx joined two OSM ways with different tags. Count the edges whose `highway` is a list: `isinstance(value, list)` is `True` when `value` is a list. Use a `for` loop over `edges["highway"]` and a counter (A3).

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
n_lists = 0
for value in edges["highway"]:
    if isinstance(value, list):
        n_lists += 1
print(n_lists, "of", len(edges))
```

48 of the 1,942 edges hold a list, for example `['track', 'unclassified']`. Lists in a column are a nuisance: `value_counts` and `groupby` treat each different list as its own type. Step 3 deals with them.
</details>

## 3. Road length by type

How many kilometres of road does the valley have, and of which kinds? Two traps lie on the way:

1. **Every road twice.** The directed graph stores each two-way road in both directions. Summing `edges` counts every kilometre twice. `ox.convert.to_undirected(G)` makes an **undirected** graph (a MultiGraph): one edge per road. Use it for lengths, and the directed graph for routing.
2. **Degrees are not metres.** The edge geometries are in EPSG:4326. Measure them in UTM 45N with `to_crs(32645)` (B4). osmnx's own `length` column is in metres too, measured on the globe; the two should agree closely, and checking that they do is cheap.

For the road type, a small function (A5) returns `highway` unchanged when it is text, and the **first** type when it is a list. `.map(function)` applies a function to every value of a column, as `.map(dictionary)` translated codes in C5.

In [ ]:
G_undirected = ox.convert.to_undirected(G)                       # each road once
roads = ox.graph_to_gdfs(G_undirected, nodes=False)                # only the edges
roads_utm = roads.to_crs(32645)                                    # metres (B4)
print(len(edges), "directed edges ->", len(roads), "roads")
print("sum of directed edges:", round(edges.to_crs(32645).length.sum() / 1000, 1), "km")
print("sum of roads:         ", round(roads_utm.length.sum() / 1000, 1), "km")
print("osmnx's length column:", round(roads["length"].sum() / 1000, 1), "km")

In [ ]:
def main_type(value):
    """The highway type of a road; if OSM gave a list of types, the first one."""
    if isinstance(value, list):
        return value[0]
    return value


roads_utm["type"] = roads_utm["highway"].map(main_type)
roads_utm["length_km"] = roads_utm.length / 1000
km_by_type = roads_utm.groupby("type")["length_km"].sum().sort_values(ascending=False)
print(km_by_type.round(1))
print("total:", round(km_by_type.sum(), 1), "km")

In [ ]:
MAP_CLASSES = [                      # (legend label, OSM types, colour, line width), smallest roads first
    ("track", ["track"], "peru", 0.7),
    ("unclassified", ["unclassified"], "dimgray", 0.7),
    ("other minor roads", ["residential", "service", "road", "living_street"], "royalblue", 1),
    ("tertiary", ["tertiary"], "darkorange", 2),
    ("secondary", ["secondary"], "firebrick", 2.5),
]

fig, ax = plt.subplots(figsize=(7, 9))
for label, types, colour, width in MAP_CLASSES:
    part = roads_utm[roads_utm["type"].isin(types)]
    part.plot(ax=ax, color=colour, linewidth=width, label=label)
ax.legend(loc="upper right", fontsize=8)
ax.set_title("Roads of the Melamchi valley by OSM type")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.ticklabel_format(style="plain", useOffset=False)
fig.text(0.01, 0.02, OSM_CREDIT, fontsize=8)
plt.show()

The directed edges add up to about **1,333.6 km**, the 971 roads to about **666.8 km**: exactly half. Summing the directed `edges` would have doubled the valley's roads. osmnx's `length` column gives about 667.5 km, within about 0.1% of the UTM lengths. Two independent measurements that agree are a good sign.

**Unclassified** roads make up about 410.5 km, more than 60% of the network. Then come **tracks** (about 113.3 km), **tertiary** roads (95.7 km) and **secondary** roads (34.8 km); the rest is small.

On the map, the main **secondary** road (red) follows the river up the valley from the town in the south-east; a second one winds through the south-west. **Tertiary** roads (orange) climb the eastern side, and a web of unclassified roads and tracks covers the hillsides. Each row of `MAP_CLASSES` is a tuple (A12) that the loop unpacks into four names. The list runs from small to big roads, so the main roads are drawn **last** and lie on top.

OSM types describe how mappers tagged a road, not how good it is today. A `secondary` road on a map can be a gravel road on the ground, and a `track` may be passable only in the dry season.

*Map data © OpenStreetMap contributors, available under the Open Database Licence (ODbL).*

### Practice

**Q1 · one line:** What percentage of the total road length is **track** or **unclassified**? Use `km_by_type` and round to 1 decimal.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
round((km_by_type["track"] + km_by_type["unclassified"]) / km_by_type.sum() * 100, 1)
```

About 78.6%: nearly four kilometres of every five are minor roads and tracks, the kind OSM maps least completely.
</details>

**Q2 · two lines:** How long is the average road segment of each type? Use `groupby("type")` on `length_km` with `.mean()`, sort from longest to shortest and round to 2 decimals.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
mean_km = roads_utm.groupby("type")["length_km"].mean()
print(mean_km.sort_values(ascending=False).round(2))
```

The single `road` segment is 2.20 km long. Among the common types, unclassified segments are the longest on average (about 0.82 km), then tertiary (0.73 km), secondary (0.57 km) and tracks (0.48 km). A segment runs from one node to the next, so long segments mean few junctions: hillside roads with few side roads. An average of one segment says little, so read such a table together with the counts.
</details>

**Q3 · three lines:** Check the two length measurements road by road. Add a column with the difference `length_km × 1000 − length` in metres, and print its smallest and largest values and the mean absolute difference (`.abs().mean()`), rounded to 1 decimal.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
roads_utm["diff_m"] = roads_utm["length_km"] * 1000 - roads_utm["length"]
print(round(roads_utm["diff_m"].min(), 1), round(roads_utm["diff_m"].max(), 1))
print(round(roads_utm["diff_m"].abs().mean(), 1))
```

The differences run from about −10.9 m to +1.4 m, and are about 0.8 m on average: the UTM length is usually a little shorter. The two methods use different models of the Earth (osmnx measures on a sphere, UTM on a flat projection), so small differences are expected. Both are fine for this lesson. Neither knows about slopes, so real driving distances up steep roads are a little longer than both.
</details>

## 4. Junctions, dead ends and connected pieces

Two simple measures describe the shape of a network.

The **degree** of a node is the number of edges that touch it. In the undirected graph, degree **1** is a **dead end**, degree **3** a junction where three roads meet, and so on. `dict(G_undirected.degree())` gives every node's degree as a dictionary, and `pd.Series(...)` turns it into a column you can count (A9).

A **connected component** is a piece of the network in which every node can reach every other node. A healthy road network is mostly one big component. `nx.connected_components(G_undirected)` yields each component as a **set** of node ids (A12). `max(components, key=len)` picks the largest: `key=len` tells `max` to compare the sets by their length.

Small, separate components have three common causes: a road cut by the edge of the download box, a mapping gap where two roads should meet but do not, or a road that really is isolated.

In [ ]:
degree = pd.Series(dict(G_undirected.degree()))
print(degree.value_counts().sort_index())

components = list(nx.connected_components(G_undirected))
largest = max(components, key=len)
sizes = sorted([len(c) for c in components], reverse=True)
print(len(components), "connected components; sizes:", sizes)
print("largest:", len(largest), "of", len(nodes), "nodes =", round(len(largest) / len(nodes) * 100, 1), "% | outside it:", len(nodes) - len(largest))

In [ ]:
nodes_utm = nodes.to_crs(32645)
nodes_utm["degree"] = degree
nodes_utm["in_largest"] = nodes_utm.index.isin(largest)

fig, ax = plt.subplots(figsize=(7, 9))
roads_utm.plot(ax=ax, color="lightgray", linewidth=0.7)
nodes_utm[nodes_utm["degree"] == 1].plot(ax=ax, color="black", markersize=4, label="dead end (degree 1)")
nodes_utm[~nodes_utm["in_largest"]].plot(ax=ax, color="red", markersize=25, label="not in the largest component")
ax.legend(loc="upper right", fontsize=8)
ax.set_title("Dead ends and separate pieces of the road network")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.ticklabel_format(style="plain", useOffset=False)
fig.text(0.01, 0.02, OSM_CREDIT, fontsize=8)
plt.show()

Most nodes have degree **3** (509 junctions) or **1** (293 **dead ends**). 13 nodes join four roads, 35 have degree 2, and 4 have degree 0: points with no road left in the box.

The network has **24** connected components. The largest holds **803** of the 854 nodes, 94.0%. The other 23 are tiny: one to five nodes each.

The map shows where they are. Many red points sit at the **edge** of the box, where the download cut roads that continue outside (the 4 lonely nodes of degree 0 are among them). A few lie inside the box: roads that, in OSM, connect to nothing. On the ground they may well connect; this is a known weakness of volunteered maps, and a reason to check before trusting a "no route".

The black dead ends are everywhere: the ends of hillside roads leading to a village, and roads cut by the box. A network with many dead ends has little **redundancy**: for many places there is only one way in.

`nodes_utm.index.isin(largest)` is `True` for each node id that is in the set `largest`, like `.isin` in B5.

*Map data © OpenStreetMap contributors, available under the Open Database Licence (ODbL).*

### Practice

**Q1 · one line:** How many nodes are dead ends? Count the nodes with degree 1 in `degree`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
(degree == 1).sum()
```

293 of the 854 nodes, about a third. Each is the end of a road: a village, a field, the edge of the box, or a road the mappers did not finish.
</details>

**Q2 · two lines:** How many kilometres of road lie **outside** the largest component? Select the roads whose start node `u` is not in `largest` (the index level `u` comes from `roads_utm.index.get_level_values("u")`) and sum their `length_km`, rounded to 2 decimals.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
outside = ~roads_utm.index.get_level_values("u").isin(largest)
print(round(roads_utm.loc[outside, "length_km"].sum(), 2))
```

About 18.88 km, under 3% of the 666.8 km. The separate pieces are small; for routing, the largest component is the network that matters. (A road joins two nodes of the same component, so testing its start node `u` is enough.)
</details>

**Q3 · three lines:** Is Melamchi town in the largest component? Find the nearest node to the OSM town point with `ox.distance.nearest_nodes(G, X=lon, Y=lat)`, using `town.geometry.x` and `town.geometry.y`, then test whether it is `in largest` and print its degree.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
town_node = ox.distance.nearest_nodes(G, X=town.geometry.x, Y=town.geometry.y)
print(town_node, town_node in largest)
print(degree[town_node])
```

Node 3476590086 is in the largest component, and it is a junction of degree 3. `X` is the **longitude** and `Y` the **latitude**, x before y as always (B1). Step 5 explains `nearest_nodes`.
</details>

## 5. The shortest route between two places

Routing needs two **nodes**. A place is a point that is usually not exactly on a road, so you **snap** it to the nearest node:

`ox.distance.nearest_nodes(G, X=lon, Y=lat, return_dist=True)` returns the nearest node and its distance in metres. **X is the longitude, Y the latitude.** Always look at the distance: a snap of a few tens of metres is fine, a few kilometres means the place has no road nearby, or something is wrong (section 9).

`ox.shortest_path(G, origin, destination, weight="length")` returns the route as a **list of node ids**. `weight="length"` makes it minimise metres; without a weight it would minimise the number of edges. Behind it is **Dijkstra's algorithm**, which grows outwards from the origin, always extending the closest unfinished node first. If no route exists, osmnx returns `None`.

`ox.routing.route_to_gdf(G, route)` returns the route's edges as a GeoDataFrame, ready to map and measure. `nx.path_weight(G, route, "length")` sums the lengths directly.

**Predict:** Helambu lies about 11 km north of Melamchi town in a straight line. How much longer will the road be?

In [ ]:
def nearest_node(place_name):
    """The graph node nearest to a named place, and the distance to it in metres."""
    point = named.loc[place_name].geometry
    return ox.distance.nearest_nodes(G, X=point.x, Y=point.y, return_dist=True)   # X = lon, Y = lat


melamchi_node, snap_m = nearest_node("Melamchi")
helambu_node, snap2_m = nearest_node("Helambu")
print("Melamchi ->", melamchi_node, round(snap_m), "m away | Helambu ->", helambu_node, round(snap2_m), "m away")

route = ox.shortest_path(G, melamchi_node, helambu_node, weight="length")
route_gdf = ox.routing.route_to_gdf(G, route).to_crs(32645)
route_km = route_gdf["length"].sum() / 1000
straight_km = named_utm.loc["Melamchi"].geometry.distance(named_utm.loc["Helambu"].geometry) / 1000
print(len(route), "nodes,", len(route_gdf), "edges")
print("route:", round(route_km, 2), "km | straight line:", round(straight_km, 2), "km | ratio:", round(route_km / straight_km, 2))
print("same as nx.path_weight:", round(nx.path_weight(G, route, "length") / 1000, 2), "km")

In [ ]:
LABELS = ["Melamchi", "Helambu", "Timbu", "Talramarang", "Dubachaur", "Gyalthum", "Sindhukot", "Sikharpur", "Sermathang", "Kutumsang"]


def base_map(ax, title):
    """Grey roads, labelled places and axis labels: the background of this lesson's maps."""
    roads_utm.plot(ax=ax, color="lightgray", linewidth=0.7)
    named_utm.loc[LABELS].plot(ax=ax, color="black", markersize=12, zorder=4)
    for name in LABELS:
        point = named_utm.loc[name].geometry
        ax.annotate(name, (point.x, point.y), xytext=(4, 4), textcoords="offset points", fontsize=8, zorder=5)
    ax.set_title(title)
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
    ax.ticklabel_format(style="plain", useOffset=False)


fig, ax = plt.subplots(figsize=(7, 9))
base_map(ax, f"Shortest route Melamchi - Helambu: {route_km:.1f} km")
route_gdf.plot(ax=ax, color="blue", linewidth=3)
fig.text(0.01, 0.02, OSM_CREDIT, fontsize=8)
plt.show()

Both places snapped to nodes close by: Melamchi about 45 m away, Helambu about 148 m away. Good snaps.

The route has 22 nodes and 21 edges, and it is about **14.48 km** long, against about 11.16 km in a straight line: a ratio of about 1.3. Valley roads wind with the river, so the road is about 30% longer than the straight line. That ratio is called **circuity**. `nx.path_weight` gives the same length, a useful check that the two ways of measuring agree.

On the map, the blue route leaves the town westwards and follows the Melamchi river up the valley, on the **valley floor**: exactly the ground D3 found changed. Step 6 puts the two together.

`base_map` draws the grey roads and ten labelled places with one call; the next maps reuse it.

*Map data © OpenStreetMap contributors, available under the Open Database Licence (ODbL).*

### Practice

**Q1 · one line:** How far is **Timbu**, a village near the top of the valley, from its nearest node? Use `nearest_node`, and round the distance to whole metres.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
nearest_node("Timbu")
```

Node 767257377, about 69 m away: Timbu is on a mapped road too. `nearest_node` returns two values, the node and the distance, shown together in brackets: a **tuple** (A12).
</details>

**Q2 · two lines:** Find the shortest route from Melamchi to **Talramarang**, on the western side of the valley, and print its length in km and its number of edges.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
route_t = ox.shortest_path(G, melamchi_node, nearest_node("Talramarang")[0], weight="length")
print(round(nx.path_weight(G, route_t, "length") / 1000, 2), "km,", len(route_t) - 1, "edges")
```

About 6.44 km in 9 edges. `nearest_node(...)` returns two values, the node and the distance; `[0]` takes the node. A route of n nodes has n − 1 edges.
</details>

**Q3 · three lines:** The route to Timbu: find it, print its length and the straight-line distance in km, and draw it in red over `base_map`. Compare it with the route to Helambu. Which way does it go?

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
timbu_node = nearest_node("Timbu")[0]
route_timbu = ox.shortest_path(G, melamchi_node, timbu_node, weight="length")
print(round(nx.path_weight(G, route_timbu, "length") / 1000, 2), "km | straight line:", round(named_utm.loc["Melamchi"].geometry.distance(named_utm.loc["Timbu"].geometry) / 1000, 2), "km")
fig, ax = plt.subplots(figsize=(7, 9))
base_map(ax, "Shortest route Melamchi - Timbu")
ox.routing.route_to_gdf(G, route_timbu).to_crs(32645).plot(ax=ax, color="red", linewidth=3)
fig.text(0.01, 0.02, OSM_CREDIT, fontsize=8)
plt.show()
```

About 33.58 km, more than twice the route to Helambu, although Timbu is only about 14.06 km from Melamchi in a straight line, about 3 km more than Helambu. The route does not follow the valley at all: it climbs the eastern hills past Dubachaur and Sermathang and comes to Timbu from the east. Why not up the valley road? In OSM, that road stops at a dead end north of Helambu, before Timbu. Is that a gap in the **road**, or a gap in the **map**? The data cannot tell you; review 4 looks closer.
</details>

## 6. Which roads lie in the flood strip?

To ask which roads the flood reached, you need D3's change area as polygons. This lesson **rebuilds** it from the two Sentinel-2 images with **exactly D3's rule**, so that it stands on its own:

1. NDVI change (after − before) below **−0.3**;
2. changed cells grouped with 8-connectivity after a one-cell `binary_dilation`, and groups of fewer than **50** cells dropped;
3. only groups whose **median height above the valley floor** (elevation minus the lowest elevation within about 500 m, from the Copernicus DEM) is below **30 m**;
4. `features.shapes` and `dissolve` to make one MultiPolygon (C7, B6).

D3 called the result *an image-based estimate, not field-checked*. It still is.

The overlay is B9 again: `roads_utm.intersection(flood_geom).length` gives, for every road, the metres that lie inside the polygons. Two cautions:

- **Position.** OSM roads are traced by volunteers, often within a few metres of the true line, sometimes tens of metres off. The change area has 20 m cells with mixed edges. A road that runs **along** the edge of the strip is very sensitive to both, so the example repeats the overlay with the polygons grown by 20 and 40 m (`buffer`, B8).
- **Bare before.** A road surface is bare, so NDVI cannot fall on it. The mask catches the vegetation **around** a road, not the road itself (D3's review 1 met the same limit).

In [ ]:
import rioxarray as rxr
from rasterio import features
from rasterio.enums import Resampling
from scipy import ndimage
from shapely.geometry import box, shape

S2_CREDIT = "Contains modified Copernicus Sentinel data 2020–2021, processed by ESA."
EIGHT = np.ones((3, 3), dtype=bool)


def open_s2(path):
    """Open a course Sentinel-2 file as reflectance (float32, NaN for nodata), with named bands (D1, D2)."""
    image = rxr.open_rasterio(path, masked=True)
    image = image.assign_coords(band=list(image.attrs["long_name"])) * 0.0001
    image.attrs = {}
    return image


def ndvi(image):
    """NDVI of an image, with NaN where NIR + red is 0 (D2)."""
    nir, red = image.sel(band="B08").values, image.sel(band="B04").values
    total = np.where(nir + red == 0, np.nan, nir + red)
    return (nir - red) / total


before = open_s2(DATA / "s2_melamchi_before.tif")
after = open_s2(DATA / "s2_melamchi_after.tif")
d_ndvi = ndvi(after) - ndvi(before)
dem = rxr.open_rasterio(DATA / "dem_kathmandu_melamchi.tif", masked=True).squeeze("band", drop=True)
elev_m = dem.rio.reproject_match(before, resampling=Resampling.bilinear).values      # C4
above_floor_m = elev_m - ndimage.minimum_filter(elev_m, size=25)                      # D3, step 5


def flood_polygons(threshold=-0.3, min_cells=50, max_above_m=30):
    """D3's change area as one dissolved polygon in UTM 45N, and its number of 20 m cells."""
    change = d_ndvi < threshold
    groups, n = ndimage.label(ndimage.binary_dilation(change, structure=EIGHT), structure=EIGHT)
    groups = np.where(change, groups, 0)
    ids, counts = np.unique(groups[groups > 0], return_counts=True)
    big = ids[counts >= min_cells]
    medians = np.array(ndimage.median(above_floor_m, labels=groups, index=big))
    mask = np.isin(groups, big[medians < max_above_m])
    shapes = features.shapes(mask.astype("uint8"), mask=mask, transform=before.rio.transform())
    polys = gpd.GeoDataFrame(geometry=[shape(geom) for geom, value in shapes], crs="EPSG:32645")
    return polys.dissolve(), int(mask.sum())


flood_gdf, flood_cells = flood_polygons(-0.3)
flood_geom = flood_gdf.geometry.iloc[0]
print(flood_cells, "cells =", round(flood_cells * 0.0004, 2), "km² | polygon area:", round(flood_geom.area / 1000000, 2), "km²")

left, bottom, right, top = before.rio.bounds()
image_box = box(left, bottom, right, top)                    # the image's rectangle (B2)
print("all roads inside the image:", bool(roads_utm.within(image_box).all()))

In [ ]:
roads_utm["inside_m"] = roads_utm.intersection(flood_geom).length
hit = roads_utm[roads_utm["inside_m"] > 0]
print(len(hit), "of", len(roads_utm), "roads touch the change area;", round(hit["inside_m"].sum() / 1000, 2), "km of road lie inside it")
print((roads_utm.groupby("type")["inside_m"].sum() / 1000).round(2).sort_values(ascending=False).head(6))

for grow_m in [0, 20, 40]:
    km_inside = roads_utm.intersection(flood_geom.buffer(grow_m)).length.sum() / 1000
    print(f"polygons grown by {grow_m:2d} m: {km_inside:5.2f} km of road inside")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 9))
base_map(ax, "Roads (red) inside D3's change area (blue)")
flood_gdf.plot(ax=ax, color="deepskyblue", zorder=2)
roads_utm.clip(flood_geom).plot(ax=ax, color="red", linewidth=2, zorder=3)
fig.text(0.01, 0.02, OSM_CREDIT + "\n" + S2_CREDIT + " Valley-floor rule: Copernicus DEM GLO-30.", fontsize=7)
plt.show()

The rebuilt mask has **8,088 cells, about 3.24 km²**, the same as D3's final mask, and the polygons have exactly that area. All 971 roads lie inside the image box, so no road falls outside the area the images cover.

**60 of the 971 roads** touch the change area, and about **10.47 km** of road lie inside it. Half of that is the **secondary** road up the valley (about 5.24 km), most of the rest **unclassified** roads (about 4.14 km). Tertiary roads, residential roads, tracks and service roads add less than 1.1 km together.

The buffer test is the warning of this step. Grow the polygons by only **20 m**, one Sentinel-2 cell, and the road length inside more than doubles to about **24.07 km**; at 40 m it is about 34.38 km. The road runs **along** the strip, metres from its edge. So the honest result is not "10.5 km of road were flooded" but: *about 10 to 34 km of today's mapped roads lie in or within 40 m of the area where vegetation was lost between 2020 and 2021, most of it the road along the valley floor*.

On the map, the red pieces follow the strip from Melamchi to the top of the valley. They are short and broken, because the road weaves in and out of the strip's edge.

`box(left, bottom, right, top)` builds the image's rectangle from its bounds; `within` checks that each road lies inside it (B7). With some library versions, `features.shapes` prints a `DeprecationWarning` about a "Memory" driver, as in C7 and D3; it is harmless.

*Map data © OpenStreetMap contributors, available under the Open Database Licence (ODbL).*
*Contains modified Copernicus Sentinel data 2020–2021, processed by ESA.*
*Elevation: Copernicus DEM GLO-30 (© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the European Union and ESA). The full licence notice is in `data/README.md`.*

### Practice

**Q1 · one line:** Which roads have the most metres inside the change area? Show the five rows of `hit` with the largest `inside_m`, with the columns `type`, `name`, `length_km` and `inside_m`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
hit.sort_values("inside_m", ascending=False)[["type", "name", "length_km", "inside_m"]].head(5)
```

The top road is a secondary road of about 1.92 km with about 1,745 m inside, almost all of it: the stretch of the valley road north of Helambu, which runs on the valley floor for its whole length. The next ones are long unclassified and secondary roads with about 620 to 900 m inside. The names are all `NaN`: many OSM roads here have no name.
</details>

**Q2 · two lines:** What **share** of each road type's length lies inside the change area? Divide the metres inside by the total metres of each type (both with `groupby("type")`) and print the percentages, rounded to 1 decimal.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
inside_by_type = roads_utm.groupby("type")["inside_m"].sum()
print((inside_by_type / (km_by_type * 1000) * 100).round(1).sort_values(ascending=False))
```

About 15.0% of the **secondary** road length lies inside the change area, but only about 1.0% of the unclassified roads and 0.7% of the tertiary roads. Service (6.1%) and residential roads (2.5%) score higher than those, but on totals of only a few kilometres. The main valley road is by far the most exposed, because it was built where the flood ran: on the valley floor. Pandas lines up the two Series by type before dividing.
</details>

**Q3 · three lines:** Extend the buffer test: loop over 0, 20, 40, 60 and 100 m, and print, for each, the road length inside the grown polygons in km and the number of roads that touch them.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
for grow_m in [0, 20, 40, 60, 100]:
    grown = flood_geom.buffer(grow_m)
    print(grow_m, round(roads_utm.intersection(grown).length.sum() / 1000, 2), roads_utm.intersects(grown).sum())
```

About 10.47, 24.07, 34.38, 41.14 and 52.24 km, touched by 60, 90, 107, 124 and 142 roads. The first 20 m add the most road; each further 20 m adds less, because the road that runs right along the strip is already inside. The lesson: the valley road is exposed whichever way you draw the line, but *how many kilometres* hangs on a few tens of metres of position, finer than either the map or the 20 m cells can promise.
</details>

## 7. Cutting the network: detour or no route

Now the **scenario**. Make a **copy** of the graph (`G.copy()`), remove the edges that cross the change area, and route again. The original graph stays untouched for comparison.

Which edges to remove is a choice, and it matters:

- **Strict:** remove every edge with **any** length inside the change area. One closed metre closes the whole edge between its two junctions, because a car cannot drive around a gap in a road. But a bridge high above the river, or a road that touches only the edge of the strip, may have survived.
- **Lenient:** remove only edges with more than **100 m** inside. Short crossings, such as bridges and fords, are assumed passable; long stretches on the valley floor are not.

Neither is the truth. Together they bracket it, as D3's three thresholds did.

The directed `edges` table has both directions of every road, so removing its rows removes both. `G.remove_edges_from(list_of_u_v_key)` takes the index of the rows to remove. If no route remains, `ox.shortest_path` returns `None`, and `nx.shortest_path` raises a `NetworkXNoPath` error, which `try`/`except` (A6) can catch.

**Predict:** in the strict scenario, will there still be a route from Melamchi to Helambu?

In [ ]:
edges_utm = edges.to_crs(32645)
edges_utm["inside_m"] = edges_utm.intersection(flood_geom).length


def cut_network(min_inside_m):
    """A copy of G without the edges that run more than min_inside_m metres through the change area."""
    closed = edges_utm[edges_utm["inside_m"] > min_inside_m].index
    cut = G.copy()
    cut.remove_edges_from(closed)
    return cut, len(closed)


G_strict, n_strict = cut_network(0)
G_lenient, n_lenient = cut_network(100)
print("closed directed edges: strict", n_strict, "| lenient", n_lenient)
print("edges left:", len(G.edges), "->", len(G_strict.edges), "(strict) and", len(G_lenient.edges), "(lenient)")

for name, graph in [("intact", G), ("strict", G_strict), ("lenient", G_lenient)]:
    path = ox.shortest_path(graph, melamchi_node, helambu_node, weight="length")
    if path is None:
        print(f"{name:8} Melamchi -> Helambu: no route")
    else:
        print(f"{name:8} Melamchi -> Helambu: {nx.path_weight(graph, path, 'length') / 1000:.2f} km")

In [ ]:
try:
    nx.shortest_path(G_strict, melamchi_node, helambu_node, weight="length")
except nx.NetworkXNoPath as error:
    print("networkx says:", error)

In [ ]:
detour = ox.shortest_path(G_lenient, melamchi_node, helambu_node, weight="length")
detour_gdf = ox.routing.route_to_gdf(G_lenient, detour).to_crs(32645)
closed_lenient = edges_utm[edges_utm["inside_m"] > 100]

fig, ax = plt.subplots(figsize=(7, 9))
base_map(ax, "Lenient scenario: closed roads (red) and the detour (blue)")
flood_gdf.plot(ax=ax, color="deepskyblue", zorder=2)
route_gdf.plot(ax=ax, color="black", linewidth=1.5, linestyle="--", zorder=3, label="route before")
closed_lenient.plot(ax=ax, color="red", linewidth=2.5, zorder=3, label="closed")
detour_gdf.plot(ax=ax, color="blue", linewidth=2.5, zorder=3, label=f"detour, {detour_gdf['length'].sum() / 1000:.1f} km")
ax.legend(loc="upper right", fontsize=8)
fig.text(0.01, 0.02, OSM_CREDIT + "\n" + S2_CREDIT, fontsize=7)
plt.show()

The strict scenario closes **120 directed edges**, that is 60 roads in both directions; the lenient one closes **42**, 21 roads.

- **Intact:** Melamchi to Helambu is about **14.48 km**.
- **Strict:** **no route**. Every way up the valley from the town crosses the change area somewhere. `nx.shortest_path` says the same with an error, `No path between 3476590086 and 767257562.`, which the `try`/`except` catches and prints.
- **Lenient:** a route exists again, about **27.25 km**, almost twice as long.

The map shows the detour. With the long valley-floor stretches closed (red), the blue route heads north from the town past Dubachaur, climbs the **hills east of the valley**, and comes back down to Helambu from the east. The black dashed line is the old route along the valley floor.

How large the disruption is depends on an assumption you cannot check from the images: whether the short crossings survived. **Report both scenarios, and say which assumption each one makes.** That is the same habit as reporting D3's area with its threshold range.

*Map data © OpenStreetMap contributors, available under the Open Database Licence (ODbL).*
*Contains modified Copernicus Sentinel data 2020–2021, processed by ESA.*

### Practice

**Q1 · one line:** How many of the closed directed edges in the strict scenario are tagged as **bridges**? Count the rows of `edges_utm` with `inside_m > 0` and `bridge == "yes"`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
((edges_utm["inside_m"] > 0) & (edges_utm["bridge"] == "yes")).sum()
```

24 of the 120 closed edges, 12 roads in both directions, include a bridge somewhere along them. These are the cases the strict scenario treats most harshly: a bridge may have stood while the fields beside it were buried, or it may have been swept away. Only field reports can say.
</details>

**Q2 · two lines:** Try a middle rule: close only edges with more than **20 m** inside. Build the graph with `cut_network(20)` and print the route length from Melamchi to Helambu, or "no route".

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
G_20, n_20 = cut_network(20)
path_20 = ox.shortest_path(G_20, melamchi_node, helambu_node, weight="length")
print(n_20, "closed:", "no route" if path_20 is None else round(nx.path_weight(G_20, path_20, "length") / 1000, 2))
```

94 directed edges closed, and **no route**: as soon as the rule closes the edges with 20 to 100 m inside, the eastern detour is gone too. The answer jumps between "twice as long" and "no way through" somewhere between 20 and 100 m. A result that flips like that deserves a sentence in any report. `"no route" if path_20 is None else ...` is a one-line `if`/`else` (a **conditional expression**).
</details>

**Q3 · three lines:** Melamchi to **Talramarang**, on the other side of the river: print the route length in km in the intact, strict and lenient networks (or "no route") in a loop.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
talramarang_node = nearest_node("Talramarang")[0]
for name, graph in [("intact", G), ("strict", G_strict), ("lenient", G_lenient)]:
    path = ox.shortest_path(graph, melamchi_node, talramarang_node, weight="length")
    print(name, "no route" if path is None else round(nx.path_weight(graph, path, "length") / 1000, 2))
```

About 6.44 km intact, and 9.35 km in **both** scenarios. The direct road runs more than 100 m through the change area, so even the lenient rule closes it, but another road, about 2.9 km longer, avoids the change area altogether. Not every place is cut off, and a place that is not cut off may still face a detour; step 8 maps both for every node.
</details>

## 8. Accessibility: how far is everything from Melamchi?

A route connects two places. **Accessibility** asks about all of them at once: how far is every node from the town, by road?

`nx.single_source_dijkstra_path_length(G, source, weight="length")` runs Dijkstra's algorithm from one node until it has reached **every** node it can. It returns a dictionary `{node: metres}`. Nodes it cannot reach are **missing** from the dictionary.

`pd.Series(dictionary)` turns it into a column indexed by node id. Assigning it to `nodes_utm` lines the values up by node id, and unreachable nodes get `NaN`. With one column per scenario, you can count, compare and map.

The town is the **source**, because it holds the market, offices and the main road out of the valley. It is a choice: accessibility from a health post or a school would be another map.

**Predict:** in the strict scenario, will most of the network still be reachable from Melamchi, or less than half of it?

In [ ]:
for name, graph in [("intact", G), ("strict", G_strict), ("lenient", G_lenient)]:
    metres = nx.single_source_dijkstra_path_length(graph, melamchi_node, weight="length")
    nodes_utm["km_" + name] = pd.Series(metres) / 1000

summary = pd.DataFrame({
    "reachable": nodes_utm[["km_intact", "km_strict", "km_lenient"]].notna().sum(),
    "within_5_km": (nodes_utm[["km_intact", "km_strict", "km_lenient"]] <= 5).sum(),
    "within_10_km": (nodes_utm[["km_intact", "km_strict", "km_lenient"]] <= 10).sum(),
    "median_km": nodes_utm[["km_intact", "km_strict", "km_lenient"]].median(),
})
print(summary.round(1))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 6.5), sharey=True)
for ax, column, title in zip(axes, ["km_intact", "km_strict", "km_lenient"], ["Intact network", "Strict scenario", "Lenient scenario"]):
    roads_utm.plot(ax=ax, color="lightgray", linewidth=0.5)
    flood_gdf.plot(ax=ax, color="deepskyblue")
    nodes_utm.plot(ax=ax, column=column, cmap="viridis_r", vmin=0, vmax=35, markersize=10, zorder=3)
    nodes_utm[nodes_utm[column].isna()].plot(ax=ax, color="red", marker="x", markersize=12, zorder=3)
    named_utm.loc[["Melamchi"]].plot(ax=ax, color="white", marker="*", markersize=250, edgecolor="black", zorder=4)
    ax.set_title(title)
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("")
    ax.ticklabel_format(style="plain", useOffset=False)
axes[0].set_ylabel("Northing (m)")
colour_scale = plt.cm.ScalarMappable(cmap="viridis_r", norm=plt.Normalize(0, 35))
fig.colorbar(colour_scale, ax=axes, shrink=0.7, label="Road distance from Melamchi (km); red x: no route")
fig.text(0.01, 0.01, OSM_CREDIT + " " + S2_CREDIT + " Star: Melamchi town.", fontsize=8)
plt.show()

Read the table row by row. In the **intact** network, **803** nodes are reachable: all of the largest component (step 4). 133 nodes lie within 5 km of the town by road, 409 within 10 km, and the median node is about 9.9 km away.

In the **strict** scenario only **324** nodes remain reachable: the town is cut off from the other **479**, the whole valley north of it and the hills on its east side. Nodes within 5 km drop from 133 to 47. The median *rises* only a little, to about 10.9 km, which shows a trap: a median of the reachable nodes ignores the nodes that are gone. Always count the unreachable ones too.

In the **lenient** scenario, 766 nodes stay reachable, but nodes within 10 km fall from 409 to 244 and the median distance grows from about 9.9 to about 14.1 km: most places can still be reached, by longer roads.

The maps (`fig` holds all three panels; `axes` is the list of the three `ax`, one per scenario) say the same at a glance. Yellow is close to the town, dark purple far away, red crosses unreachable. In the strict scenario, red covers almost everything north and east of the town. In the lenient scenario, colours turn much darker south-east of the town, around Sikharpur: the same places, further away. The red crosses in the intact map are the 51 nodes outside the largest component (step 4).

`plt.cm.ScalarMappable` makes a colour bar for a colour scale that is shared by all three panels (`vmin=0, vmax=35` in every one), so equal colours mean equal distances.

One more caution: the network ends at the box. The town's main road continues **south**, out of the box, and roads outside it may offer detours that this analysis cannot see. "Unreachable" here means **unreachable inside the box**.

*Map data © OpenStreetMap contributors, available under the Open Database Licence (ODbL).*
*Contains modified Copernicus Sentinel data 2020–2021, processed by ESA.*

### Practice

**Q1 · one line:** How many nodes does Melamchi lose in the lenient scenario? Subtract the reachable counts in `summary`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
summary.loc["km_intact", "reachable"] - summary.loc["km_lenient", "reachable"]
```

37 nodes. Even when all short crossings stay open, a few places are reachable only along the closed stretches of the valley floor.
</details>

**Q2 · two lines:** How much longer does the road to each node become in the lenient scenario? Add a column `extra_km` = `km_lenient` − `km_intact` and print its median and its maximum, rounded to 1 decimal.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
nodes_utm["extra_km"] = nodes_utm["km_lenient"] - nodes_utm["km_intact"]
print(round(nodes_utm["extra_km"].median(), 1), round(nodes_utm["extra_km"].max(), 1))
```

A median of 0.0 km, but a maximum of about 25.1 km. Most reachable nodes are not affected at all: their shortest road never touched the change area. The nodes that are affected pay a lot. A median hides that; look at the maximum and at the map too.
</details>

**Q3 · three lines:** Make a table of the **named** places: for each name in `LABELS`, the nearest node and its road distance in km in the three networks. Use `nearest_node` in a loop and `nodes_utm.loc[node, ["km_intact", "km_strict", "km_lenient"]]`.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
rows = {}
for name in LABELS:
    node = nearest_node(name)[0]
    rows[name] = nodes_utm.loc[node, ["km_intact", "km_strict", "km_lenient"]]
print(pd.DataFrame(rows).T.astype(float).round(2))
```

Only Sindhukot (13.47 km) and Talramarang (6.44 km, then 9.35 km) keep a route in the strict scenario; the other seven places, from Sikharpur 3.18 km away to Timbu 33.58 km away, have none. In the lenient scenario every place has a route again, but **Sikharpur** is then about 25.28 km away: its short road from the town runs more than 100 m through the change area south of the town, and the next way round is long. Helambu doubles (14.48 to 27.25 km); Dubachaur, Timbu and Sermathang do not change. `pd.DataFrame(rows)` builds a table with one column per place; `.T` turns it so that each place is a row (D4), and `.astype(float)` makes sure that all columns hold numbers before rounding.
</details>

## 9. When it breaks: latitude and longitude swapped

The most common routing bug gives **no error message** at all. Swap longitude and latitude in `nearest_nodes`, and osmnx still returns a node: the one nearest to a point near the North Pole. Everything after that runs, and the route is nonsense.

The defence is the check from step 5: **look at the snap distance**, and make the check automatic with `assert` (A12). Run the cell.

In [ ]:
wrong_node, wrong_m = ox.distance.nearest_nodes(G, X=town.geometry.y, Y=town.geometry.x, return_dist=True)   # X and Y swapped!
print("nearest node:", wrong_node, "| distance:", round(wrong_m / 1000), "km")
assert wrong_m < 500, f"The nearest node is {wrong_m / 1000:.0f} km away: check that X is the longitude and Y the latitude"

The first line runs without complaint and returns a real node of the valley. Only the distance gives it away: about **6,641 km**. With X = 27.83 and Y = 85.58, osmnx looked for the node nearest to a point at 85.58° **N**, far up in the Arctic; every node of the valley is about the same, enormous distance away.

The last line of the error says:

`AssertionError: The nearest node is 6641 km away: check that X is the longitude and Y the latitude`

- **AssertionError**: an `assert` found its condition `False` and stopped the notebook, on purpose.
- The **message** is yours. Write messages that say what to check, not only that something failed.

The same symptom, a nearest node kilometres away, appears in a second classic mix-up: a graph projected to UTM (`ox.project_graph`) searched with longitude and latitude in degrees, or the other way round. X and Y must be in the **CRS of the graph**: `G.graph["crs"]` tells you which.

**Try it:** swap `X` and `Y` back (`X=town.geometry.x, Y=town.geometry.y`) and run the cell again. The distance becomes about 45 m and the `assert` passes silently.

## 10. What this can and cannot tell you

**What it can tell you**

- How the mapped road network of the Melamchi valley is built: about 667 km of mostly minor roads and tracks, one main road along the valley floor, many dead ends, and little redundancy.
- Which of today's mapped roads run through the area where D3 found vegetation lost between October 2020 and November 2021: about 10.5 km, mostly the valley road, and between about 10 and 34 km if the edge of the change area or the roads are off by up to 40 m.
- How sensitive access is to the valley floor. If every road through the change area were closed, Melamchi town could reach only 324 of its 803 network nodes inside the box, and there would be no road to Helambu. If short crossings stayed open, most places stay reachable, some by detours of up to about 25 km.

**What it cannot tell you**

- **Which roads were really cut, and for how long.** The change area is an image-based estimate from two dates, without field checks (D3). The closures are a scenario. Reports, photos and field visits are needed to say what happened.
- **The road network of 2021.** The roads come from OpenStreetMap as downloaded in **2026**. Some roads may have been rebuilt on new alignments after the flood, some may have been added to the map later, and some may never have been mapped.
- **Whether the map is complete.** OSM completeness varies, especially for tracks. A missing link can create a false "no route", and a road that exists only on the map can create a false detour. The gaps in the valley road below Timbu (review 4) are such a question.
- **Anything outside the box.** The network was cut at the edge of the download box, so detours through the next valley are invisible.
- **Travel time.** Distance is not time. A detour on a steep, unpaved track takes far longer than the same distance on the valley road, and in the monsoon it may not be passable at all.

When you report this kind of analysis, say all of it: the data and their dates, the rule that closed the roads, both scenarios, and the limits. A road network with a stated scenario is a planning tool; it is not a damage report.

*Map data © OpenStreetMap contributors, available under the Open Database Licence (ODbL).*
*Contains modified Copernicus Sentinel data 2020–2021, processed by ESA.*

## Mixed review

These exercises combine everything in this lesson. Each one needs 5 to 10 lines. Plan in words first, then write the code.

### Review 1 · How much do D3's choices matter for the roads?

Use `flood_polygons(threshold)` to rebuild the change area for D3's three thresholds, −0.2, −0.3 and −0.4. For each, print the area in km² (`flood_cells × 0.0004`), the road length inside it in km, and the number of roads that touch it, rounded to 2 decimals. Collect the results in a list of dictionaries and show them as a DataFrame.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
rows = []
for threshold in [-0.2, -0.3, -0.4]:
    polys, cells = flood_polygons(threshold)
    geom = polys.geometry.iloc[0]
    rows.append({"threshold": threshold, "area_km2": cells * 0.0004,
                 "road_km_inside": roads_utm.intersection(geom).length.sum() / 1000,
                 "roads_touched": int(roads_utm.intersects(geom).sum())})
print(pd.DataFrame(rows).round(2))
```

About 4.87, 3.24 and 2.24 km² of change area hold about 17.95, 10.47 and 6.69 km of road, touched by 100, 60 and 47 roads. The road length follows the area, and grows even faster: the loosest threshold adds a fringe along the strip (D3), next to the road. Together with the buffer test of step 6, this is the range to report. Every choice made in D3 travels into this lesson's answers.
</details>

### Review 2 · Named roads in the change area

Which **named** roads have the most metres inside the change area? Fill missing names in `hit` with "(no name)" (`.fillna`), then sum `inside_m` per name with `groupby`, sort, and print the result in km, rounded to 2 decimals. What share of the metres inside belongs to roads without a name?

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
by_name = hit.assign(name=hit["name"].fillna("(no name)")).groupby("name")["inside_m"].sum() / 1000
by_name = by_name.sort_values(ascending=False)
print(by_name.round(2))
print("no name:", round(by_name["(no name)"] / by_name.sum() * 100, 1), "%")
```

"(no name)" leads by far with about 8.38 km; Melamchi Road has about 0.81 km inside, Chanaute-Ichok Road about 0.51 km, Bhothang Road about 0.43 km and Ambathan road about 0.34 km. About 80.1% of the metres inside belong to roads that OSM does not name, including most of the secondary road up the valley. A report that says "Melamchi Road lies in the change area for 0.8 km" would badly understate the exposure; name tags are optional in OSM, so group by type or by geometry, not by name. `.assign(name=...)` returns a copy with the column replaced, leaving `hit` unchanged.
</details>

### Review 3 · Save the scenario for QGIS

Save the roads that touch the change area, with only the columns `type`, `length_km`, `inside_m` and `geometry`, and a new column `closed_lenient` (`True` where `inside_m > 100`), to `d05_roads_in_change_area.gpkg` in `../outputs`, layer `roads`. Also save `flood_gdf` as layer `change_area`. Read the roads back and print the number of rows and how many are closed in the lenient scenario.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
to_save = hit[["type", "length_km", "inside_m", "geometry"]].copy()
to_save["closed_lenient"] = to_save["inside_m"] > 100
path = OUT / "d05_roads_in_change_area.gpkg"
to_save.reset_index(drop=True).to_file(path, layer="roads", driver="GPKG")
flood_gdf.to_file(path, layer="change_area", driver="GPKG")

back = gpd.read_file(path, layer="roads")
print(len(back), "roads,", int(back["closed_lenient"].sum()), "closed in the lenient scenario")
```

60 roads, 21 of them closed in the lenient scenario, as in step 7. Only plain columns are saved: the graph's columns such as `osmid` and `highway` can hold Python **lists**, which a GeoPackage cannot store. `reset_index(drop=True)` drops the (u, v, key) index, which the saved layer does not need. In QGIS you can now style the two scenarios side by side.
</details>

### Review 4 · A gap in the road, or in the map?

Step 5 found that the route to Timbu goes over the eastern hills. Look at the top of the valley. Collect the nodes of all **secondary** roads (the `u` and `v` index levels of `roads_utm`, joined with the set operator `|`, A12), keep those with degree 1, and select the ones in the box x 355–359 km, y 3,091–3,093 km with `.cx` (B5). For each, print its northing, whether it is in the largest component (`in_largest`, step 4), and the distance to the nearest **other** of these dead ends. Then map the top of the valley with `base_map`, the change area, the secondary roads in red, these dead ends in black, and `ax.axis([355000, 359000, 3090000, 3095500])`.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
secondary = roads_utm[roads_utm["type"] == "secondary"]
secondary_nodes = set(secondary.index.get_level_values("u")) | set(secondary.index.get_level_values("v"))
ends = nodes_utm[nodes_utm.index.isin(secondary_nodes) & (nodes_utm["degree"] == 1)]
gap_ends = ends.cx[355000:359000, 3091000:3093000].copy()
gap_ends["northing_m"] = gap_ends.geometry.y.round()
gap_ends["nearest_end_m"] = [round(gap_ends.geometry.drop(n).distance(p).min()) for n, p in gap_ends.geometry.items()]
print(gap_ends[["northing_m", "in_largest", "nearest_end_m"]].sort_values("northing_m"))

fig, ax = plt.subplots(figsize=(7, 8))
base_map(ax, "Top of the valley: the secondary road and its dead ends")
flood_gdf.plot(ax=ax, color="deepskyblue", zorder=2)
secondary.plot(ax=ax, color="red", linewidth=2, zorder=3)
gap_ends.plot(ax=ax, color="black", markersize=40, zorder=4)
ax.axis([355000, 359000, 3090000, 3095500])
fig.text(0.01, 0.02, OSM_CREDIT + " " + S2_CREDIT, fontsize=7)
plt.show()
```

Four dead ends of the secondary road lie within about 1 km of each other. Going north: the road from Helambu ends at about 3,091,674 m. Only **34 m** away, at about 3,091,687 m, a short separate piece begins, which is **not** in the largest component: in OSM the two were never joined. That piece ends at about 3,092,162 m (the 499 m in its row is the straight distance to its own other end), and the road towards Timbu starts again **539 m** further on, at about 3,092,491 m. On the map, the gap lies in the change area.

The 34 m gap looks like a **mapping error**: two drawn lines that should share a node. Anyone who fixes it in OpenStreetMap improves every route planner that uses OSM. The 539 m gap is a real question. Was the road there swept away in 2021 and not rebuilt by 2026? Was it never built, or never mapped? The data cannot tell you; a local mapper, a field visit or recent images can. Until then, the 33.58 km route to Timbu is an answer about the **map**, not necessarily about the valley. `gap_ends.geometry.drop(n)` leaves out the dead end itself, so `.min()` finds the nearest *other* one.
</details>

## Recap

- **OpenStreetMap** is volunteered map data with **tags** (`key=value`, such as `highway=secondary`). It is open under the ODbL (credit "© OpenStreetMap contributors"), and its completeness varies, especially for tracks. Check names, dates and gaps before you trust it.
- A road network is a **graph**: **nodes** (junctions, dead ends) and **edges** (road segments with `length`). osmnx loads it as a MultiDiGraph in which every two-way road appears **twice**; use `ox.convert.to_undirected` before summing lengths, and measure in UTM 45N.
- **Degree** finds dead ends and junctions; **connected components** find separate pieces, often caused by the download box or mapping gaps. Snap places to nodes with `ox.distance.nearest_nodes(G, X=lon, Y=lat, return_dist=True)` and **check the distance**; then `ox.shortest_path(..., weight="length")` finds the shortest route.
- Roads ∩ change area (B9) shows the exposure: about 10.5 km of today's mapped roads lie inside D3's change area, mostly the valley road, and the number is very sensitive to positions (24 km with a 20 m buffer). Report it as a range.
- **Disruption** is a scenario: remove the closed edges from a **copy** of the graph and route again. A strict rule leaves no road to Helambu, a lenient one a detour of almost twice the length. **Accessibility** with `nx.single_source_dijkstra_path_length` maps the consequences for every node. State the rule, show both scenarios, and say what the data cannot tell.

### Words you learned

| Word | Meaning |
| --- | --- |
| OpenStreetMap (OSM) | A free map of the world, drawn and updated by volunteers |
| tag (key=value) | A label on an OSM feature, such as `highway=track` or `name:en=Melamchi` |
| ODbL | The Open Database Licence of OSM data; it asks for the credit "© OpenStreetMap contributors" |
| graph / network | A set of nodes and the edges that link them |
| node | A point of a graph; in a road network a junction, a dead end or the end of a road |
| edge | A link between two nodes; here a road segment from one node to the next |
| directed / undirected graph | Edges with a direction (from u to v) / edges without one |
| MultiDiGraph | A directed graph that may hold several edges between the same two nodes |
| GraphML | A text file format for saving graphs |
| simplification (osmnx) | Removing nodes where nothing happens, so that one edge runs from junction to junction |
| degree | The number of edges that touch a node |
| dead end | A node of degree 1: a road that goes no further |
| connected component | A piece of a network in which every node can reach every other node |
| redundancy | Having more than one route between places, so that one closure does not cut them off |
| snapping | Moving a place to the nearest node of the network before routing |
| weight | The edge attribute a route minimises, here `length` in metres |
| Dijkstra's algorithm | The classic method to find shortest routes, growing outwards from the origin |
| circuity | Route length divided by straight-line distance |
| scenario | A stated "what if" assumption, such as closing every road through the change area |
| disruption | Routes that become longer or impossible when parts of a network are closed |
| accessibility | How easily places can be reached; here the road distance from Melamchi town to every node |
| conditional expression | A one-line if/else: `a if condition else b` |

## Next

**D6 · Image time series and data cubes:** this lesson and D3 compared two dates. D6 opens a **data cube** of 112 Sentinel-2 dates from 2019 to 2023, and follows the valley, and its flood-hit floor, through five years and five monsoons.

<!-- pygile-nav -->
**PyGILE Course** · lesson 38 of 39 · [Course home](../README.md)  
← Previous: [D4 · Land-cover classification: forest, fields, sediment and water](D04_Land_cover_classification.ipynb) · Next: [D6 · Image time series and data cubes: five years of the Melamchi valley](D06_Image_time_series_and_data_cubes.ipynb) →